# Jupyter Storage Explorer

A made-up data lake of a coffee roaster, held in the kernel's memory. Run the cells (**Run ▸ Run All Cells**), then try:

- **Browse**: click folders in the tree; double-click a folder in the list to go into it.
- **Preview**: select a file, then **Preview** at the bottom right. In `curated/`, `customers` is a Delta Lake table: **Query data** opens it in the data browser.
- **Right-click** a file: **Python code** to load it, Copy URL, Copy, Cut, Paste, Rename, Properties...
- **Move and copy**: drag files onto a folder, or copy them into the empty **Scratch** storage (the drop-down above the tree).
- **Folders**: moving, copying, renaming or deleting them takes **Unsafe mode**, the switch in the toolbar, which turns itself off after one operation.
- **Open in a tab**: the **Storage Explorer** item of JupyterLab's launcher (**+** above).

See [Using the explorer](https://github.com/sugibuchi/jupyter-storage-explorer/blob/main/docs/storage-explorer.md#using-the-explorer).

In [ ]:
import demo_data
from jupyter_storage_explorer import StorageExplorer

StorageExplorer(
    filesystems=demo_data.filesystems(),
    actions=demo_data.actions(),
    plugin_options=demo_data.PLUGIN_OPTIONS, height="450px"
)

## DataExplorer

Any dataframe, here the orders as a polars DataFrame: page through, sort, filter and choose columns, then get the result back with `explorer.result()`.

In [ ]:
import polars as pl
from jupyter_storage_explorer import DataExplorer

orders = pl.read_parquet(demo_data.local_copy() / "sales-lake/curated/orders.parquet")
explorer = DataExplorer(orders)
explorer

`DataExplorer` has `on_change()` too: filter the orders above, e.g. **status** is `refunded`, and the chart of orders by channel follows.

In [ ]:
import ipywidgets
import matplotlib.pyplot as plt

by_channel = ipywidgets.Output()


def draw_channels(explorer):
    with by_channel:
        by_channel.clear_output(wait=True)
        counts = explorer.result(all_columns=True).group_by("channel").len().sort("channel")
        fig, ax = plt.subplots(figsize=(5, 2.5))
        ax.bar(counts["channel"], counts["len"])
        ax.set_ylabel("Orders")
        plt.show()


explorer.on_change(draw_channels)
draw_channels(explorer)
by_channel

## Your code follows the explorer

`on_change()` calls a function when the folder or the selection changes (`folder_url`, `selected_urls`); `on_query_change()` when the table in the data browser changes, filters included (`query_result()`). Here, an explorer whose preview panel starts open (`preview_panel="open"`), and a chart of revenue by month of the table you click: open `curated`, click `orders.parquet`, then filter it in the data browser, e.g. **channel** is `wholesale`, and the chart follows.

In [ ]:
import ipywidgets
import matplotlib.pyplot as plt

lake = StorageExplorer(
    filesystems=demo_data.filesystems(),
    actions=demo_data.actions(),
    plugin_options=demo_data.PLUGIN_OPTIONS, height="300px", preview_panel="open"
)
selection = ipywidgets.Label()
chart = ipywidgets.Output()


def show_selection(explorer):
    selection.value = "Selected: " + (", ".join(explorer.selected_urls) or "nothing")


def draw(explorer):
    with chart:
        chart.clear_output(wait=True)
        result = explorer.query_result()
        if result is None or not {"order_ts", "amount"} <= set(result.collect_schema().names()):
            print("Click a table of orders: curated/orders.parquet, or a CSV file in landing/orders.")
            return
        monthly = (
            result.group_by(month=pl.col("order_ts").dt.strftime("%Y-%m"))
            .agg(pl.col("amount").sum())
            .sort("month")
            .collect()
        )
        fig, ax = plt.subplots(figsize=(8, 2.5))
        ax.bar(monthly["month"], monthly["amount"])
        ax.set_ylabel("Revenue (EUR)")
        ax.tick_params(axis="x", labelrotation=90)
        plt.show()


lake.on_change(show_selection)
lake.on_query_change(draw)
show_selection(lake)
draw(lake)
display(lake, selection, chart)

## Transfer policy

Rules for where files may go, e.g. to keep raw data in the lake. Here, files of `landing/` may not be copied or moved to **Scratch**: copy `landing/products.tsv` and paste it in Scratch (the drop-down above the tree), and the status bar says why it is not pasted; `curated/orders.parquet` is. See [Transfer policy](https://github.com/sugibuchi/jupyter-storage-explorer/blob/main/docs/storage-explorer.md#transfer-policy).

In [ ]:
StorageExplorer(
    filesystems=demo_data.filesystems(),
    actions=demo_data.actions(),
    plugin_options=demo_data.PLUGIN_OPTIONS,
    height="300px",
    transfer_policy=[
        {"rule": "deny", "src": "memory:///sales-lake/landing/.*", "dest": "memory:///scratch/.*"},
    ],
)